In [137]:
import fitz

In [138]:
# give path to any PDF inside /data
pdf_path =  "../data/MIV 2030 Report.pdf"
#pdf_path =  "C:\\Users\\Dakshyani Murari\\OneDrive\\Desktop\\Maritime_RAG\\data\\handwritten notes.pdf"


In [139]:

import fitz  # PyMuPDF

documents = []

doc = fitz.open(pdf_path)

for i, page in enumerate(doc):
    text = page.get_text("text")  # 🔥 better extraction
    
    if text:
        documents.append({
            "text": text,
            "page": i + 1
        })

In [140]:
print(documents[0]["text"][:1000])

<TtPT. TTtcT uRdgH sfhf vieFTPf *T5TTcPI 
TfR tT t R ^ R
Ministry of Ports, Shipping and Waterways 
Government of India
SAGARMALA
PORT-LED PROSPERITY
MARITIME INDIA VISION 2030



In [141]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [142]:
chunks = []

for doc in documents:
    split_chunks = chunk_text(doc["text"])
    
    for chunk in split_chunks:
        chunks.append({
            "content": chunk,
            "page": doc["page"]
        })

In [143]:
texts = [c["content"] for c in chunks]
metadatas = [{"page": c["page"]} for c in chunks]

In [144]:
print(chunks[1])


{'content': 'Prime Minister\nMESSAGE\nI am pleased to learn that the Ministry of Ports, Shipping & Waterways has \nprepared ‘Maritime India Vision 2030’ (MIV 2030). The document is a \nblueprint to accelerate the growth of our maritime sector over the next \ndecade.\nWe have been blessed with a rich maritime heritage. To shape our maritime \nprowess into a robust engine of the nation’s development, we have given \ntop priority to port-led development. We firmly believe that the immense \npotential of our coastline st', 'page': 3}


In [145]:
print(len(chunks))

1787


In [146]:
#print(chunks[5])

In [147]:
import re

def clean_text(text):
    # remove weird characters
    text = re.sub(r'[^a-zA-Z0-9.,()\-:\n ]', ' ', text)
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    # optional: remove very short noisy lines
    lines = text.split('\n')
    lines = [line.strip() for line in lines if len(line.strip()) > 30]
    
    return "\n".join(lines)

In [148]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(full_text)

print(len(chunks))
print(chunks[10])

1644
level of growth in the new decade.
(Mansukh Mandaviya)
Date: February 12, 2021 
Place: New Delhi
SA G A R M A LA #Mai Bhi Mohan
Room No. 201, Transport Bhawan, New Delhi-110001, Ph. : 011 -23717422, 23717423, 23717424 Fax : 011 -23724653
s5T.
DR. SANJEEV RAN JAN
M R ?  TTZepr? 
GOVERNMENT OF INDIA
e rlrR
SECRETARY
«Trt T Va TV$
o TcH TR f ^ToTTeRI 
MINISTRY OF PORTS,  
SHIPPING AND WATERWAYSV
PREFACE
Maritime India Vision 2030 (MIV 2030) has been prepared after extensive consultations


In [149]:
cleaned_text = clean_text(full_text)

chunks = splitter.split_text(cleaned_text)

print(chunks[10])

Ph. : 011 -23717422, 23717423, 23717424 Fax : 011 -23724653 s5T. DR. SANJEEV RAN JAN M R TTZepr GOVERNMENT OF INDIA e rlrR SECRETARY Trt T Va TV o TcH TR f ToTTeRI MINISTRY OF PORTS, SHIPPING AND WATERWAYSV PREFACE Maritime India Vision 2030 (MIV 2030) has been prepared after extensive consultations with public and private sector stakeholders. 14 Thrust area groups across various maritime sectors were constituted at the start of the exercise, to discuss and identify initiatives and targets that


In [150]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sample = chunks[10]

embedding = model.encode(sample)

print(len(embedding))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2815.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384


In [151]:
'''from sklearn.metrics.pairwise import cosine_similarity

query = "What is Maritime India Vision 2030?"
query_embedding = model.encode(query)

# check first 20 chunks
scores = []

for i in range(20):
    chunk_embedding = model.encode(chunks[i])
    sim = cosine_similarity([query_embedding], [chunk_embedding])[0][0]
    scores.append((i, sim))

# sort by similarity
scores = sorted(scores, key=lambda x: x[1], reverse=True)

print(scores[:5])'''

'from sklearn.metrics.pairwise import cosine_similarity\n\nquery = "What is Maritime India Vision 2030?"\nquery_embedding = model.encode(query)\n\n# check first 20 chunks\nscores = []\n\nfor i in range(20):\n    chunk_embedding = model.encode(chunks[i])\n    sim = cosine_similarity([query_embedding], [chunk_embedding])[0][0]\n    scores.append((i, sim))\n\n# sort by similarity\nscores = sorted(scores, key=lambda x: x[1], reverse=True)\n\nprint(scores[:5])'

In [152]:
best_index = scores[0][0]
print(chunks[best_index])

TtPT. TTtcT uRdgH sfhf vieFTPf T5TTcPI TfR tT t R R Ministry of Ports, Shipping and Waterways Government of India SAGARMALAP O R T-L E D P R O S P E R ITY MARITIME INDIA VISION 2030 Prime Minister MESSAGE I am pleased to learn that the Ministry of Ports, Shipping Waterways has prepared Maritime India Vision 2030 (MIV 2030). The document is a blueprint to accelerate the growth of our maritime sector over the next decade. We have been blessed with a rich maritime heritage. To shape our maritime


In [153]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [154]:
'''embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

db = Chroma.from_texts(
    chunks,
    embedding=embedding_model
)'''

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

db = Chroma.from_texts(
    texts,                # ✅ ONLY text
    embedding=embedding_model,
    metadatas=metadatas  # ✅ ADD THIS
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2198.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [155]:
results = db.similarity_search("ports", k=2)

print(results[0].metadata)

{'page': 83}


In [156]:
query = "What is Maritime India Vision 2030?"

results = db.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---\n")
    print(doc.page_content)


--- Result 1 ---

ages continuing the 
improvements in sectoral performance and in that regard formulated an extensive 
exercise to define Maritime India Vision 2030. The exercise involved extensive 
consultations and brainstornhing discussions with both public & private sector 
stakeholders to ensure that the vision captures initiatives that are implementable in a 
time-bound manner.
Maritime India Vision 2030 has identified 150+ initiatives across ports, 
shipping & waterways sub-sectors which will propel India

--- Result 2 ---

ages continuing the 
improvements in sectoral performance and in that regard formulated an extensive 
exercise to define Maritime India Vision 2030. The exercise involved extensive 
consultations and brainstornhing discussions with both public & private sector 
stakeholders to ensure that the vision captures initiatives that are implementable in a 
time-bound manner.
Maritime India Vision 2030 has identified 150+ initiatives across ports, 
shipping & waterw

In [165]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")  # because you're inside notebooks/



True

In [166]:
from groq import Groq

client = Groq(api_key=groq_api_key)

In [167]:
# ===== SETUP CELL =====
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv("../.env")

groq_api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

In [160]:
def rewrite_query(query):
    prompt = f"""
    Rewrite the user query to be more specific and suitable for document search.

    Query: {query}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [161]:
def ask_question(query):

    # 1️⃣ rewrite query
    improved_query = rewrite_query(query)

    # 2️⃣ retrieval (MUST come before using results)
    results = db.similarity_search(improved_query, k=5)

    # 3️⃣ extract pages
    pages = [doc.metadata["page"] for doc in results]
    unique_pages = sorted(set(pages))

    # 4️⃣ context
    context = "\n\n".join([doc.page_content for doc in results])

    # 5️⃣ prompt
    prompt = f"""
    You are an assistant answering questions from a document.

    Rules:
    - Answer ONLY from context
    - If not found, say "Not found in document"
    - Be clear and structured

    Context:
    {context}

    Question:
    {query}
    """

    # 6️⃣ LLM call
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content

    # 7️⃣ return with sources
    return answer + f"\n\n📄 Sources: Pages {unique_pages}"

In [162]:
print(ask_question("explain me waste to wealth"))

**Explanation of Waste to Wealth**

Waste to Wealth refers to the process of converting waste materials into valuable products or useful by-products that can benefit both the public and the environment.

Some key points related to Waste to Wealth mentioned in the document are:

- Conducting active recycling of waste material and re-usage of recycled material (Section 9.13.1, Q4, 2022)
- Segregating solid waste, plastic, and biodegradable materials (Section 9.13.2, Q4, 2022)
- Reusing bio-degradable/plastic materials in civil construction and other purposes (Section 9.13.3, Q4, 2023)
- Promoting the use of biodegradable waste for the production of useful by-products for public and environmental use (Section 9.13.4, Q4, 2025)

Additionally, the document mentions employing sustainable dredging disposal mechanisms (Section 9.14) and promoting recycling in port areas through the use of litter and recycling bins (Section 9.6).

📄 Sources: Pages [236, 293]
